# Moving application of ShellSIM from 1D time series data to complex 4D gridded dataset

Needed Variables 

T_timeseries=  Temperature Time series  
S_timeseries=  Practical Salinity Time series  
Chl_timeseries= Chlorophyll Time series  
POC_timeseries= Particulate Organic Carbon Time series  
POM_timeseries=  Particulate Organic Matter Time series  
TPM_timeseries= Total Particulate Matter   Time series  

In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import pyfabm
import os
from dask.diagnostics import ProgressBar
import warnings
import gc

In [2]:
# Define the integration time period 
start = '02-06-2021'
end = '04-06-2021'
time_horizon = pd.date_range(start=start, end=end, freq='1d')
time_horizon_len = len(time_horizon)

In [3]:
#  Lat lon chunking method
chunking_config={'time': -1, 'latitude': 80, 'longitude': 110}

def load_nc_file(file_path, var_name_in_file, chunking_config=chunking_config):
    """
    Loads a NetCDF file or creates a fake one if not found.
    Args:
        file_path (str): Path to the .nc file.
        var_name_in_file (str): The name of the variable to use.
        chunking_config (dict): Dask chunking configuration.
    Returns:
        xr.Dataset
    """
    fake_filename = f'{var_name_in_file}_gridded_FAKE.nc'
    if os.path.exists(file_path):
        print(f"Successfully loaded: {var_name_in_file}\n")
        return xr.open_dataset(file_path, chunks=chunking_config)

    if os.path.exists(fake_filename):
        print(f"Using existing fake dataset: {fake_filename}\n")
        return xr.open_dataset(fake_filename, chunks=chunking_config)

    print(f"Creating fake gridded data for variable '{var_name_in_file}'...")
    fake_data = np.random.rand(time_horizon_len, 100, 100)
    fake_lats = np.linspace(40, 50, 100)
    fake_lons = np.linspace(-10, 0, 100)

    fake_gridded_dataset = xr.Dataset(
        {var_name_in_file: (['time', 'latitude', 'longitude'], fake_data)},
        coords={'time': time_horizon, 'latitude': fake_lats, 'longitude': fake_lons}
    )
    fake_gridded_dataset.to_netcdf(fake_filename)
    print(f"Saved fake dataset to: {fake_filename}\n")
    return xr.open_dataset(fake_filename, chunks=chunking_config)
        

In [4]:
# Define datasets to be read and used for model
poc_file_path="/home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_BIO_BGC_3D_REP_015_010/cmems_obs-mob_glo_bgc-chl-poc_my_0.25deg_P7D-m_poc_13.00W-42.00E_30.00N-70.00N_0.00-1000.00m_2021-06-01-2021-06-30_3f1f32d8adbe3d686e11d7d4a40be7bb.nc"
poc_var_name = 'poc'  # variable name inside poc.nc

salinity_file_path="/home/jovyan/wise_data_store/hda_download/MULTIOBS_GLO_PHY_S_SURFACE_MYNRT_015_013/cmems_obs-mob_glo_phy-sss_my_multi_P1D_sos-sos_error_13.00W-42.00E_30.00N-70.00N_2021-06-01-2021-06-30_9fe69177ea51eb9c1d733df494870646.nc"
salinity_var_name = 'sos' # variable name inside salinity.nc

# Fake data generation to be triggered in the 'except' block of load_nc_file if path doesn't exist
temp_file_path = 'non_existent_temp.nc'
temp_var_name = 'temperature' # variable name inside temperature.nc

chl_file_path = 'non_existent_chl.nc'
chl_var_name = 'chl'

pom_file_path = 'non_existent_pom.nc'
pom_var_name = 'pom'

tpm_file_path = 'non_existent_tpm.nc'
tpm_var_name = 'tpm'


In [5]:
# Load all datasets
print("Loading datasets...\n")
ds_poc = load_nc_file(poc_file_path, poc_var_name)
ds_sal = load_nc_file(salinity_file_path, salinity_var_name)
ds_temp = load_nc_file(temp_file_path, temp_var_name)
ds_chl = load_nc_file(chl_file_path, chl_var_name)
ds_pom = load_nc_file(pom_file_path, pom_var_name)
ds_tpm = load_nc_file(tpm_file_path, tpm_var_name)

Loading datasets...

Successfully loaded: poc

Successfully loaded: sos

Creating fake gridded data for variable 'temperature'...
Saved fake dataset to: temperature_gridded_FAKE.nc

Creating fake gridded data for variable 'chl'...
Saved fake dataset to: chl_gridded_FAKE.nc

Creating fake gridded data for variable 'pom'...
Saved fake dataset to: pom_gridded_FAKE.nc

Creating fake gridded data for variable 'tpm'...
Saved fake dataset to: tpm_gridded_FAKE.nc



In [6]:
# Align coordinates - using POC data as reference
# Grid alignment using ds_poc as the reference grid and interpolating all other variables to it, 
# as a way to handle mismatched grids.
print("Aligning coordinates...")
ref_lats = ds_poc.latitude
ref_lons = ds_poc.longitude


Aligning coordinates...


In [7]:
interp_kwargs = {'fill_value': 'extrapolate'}

# For 4D variables: Select first depth values -  Select a single depth level and interpolate 
# use .sel(depth=0, method='nearest') to grab the surface layer
poc_daily = ds_poc[poc_var_name].sel(depth=0, method='nearest').interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
sal_daily = ds_sal[salinity_var_name].sel(depth=0, method='nearest').interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)


# Process 3D variables (datasets with no depth)
temp_daily = ds_temp[temp_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
chl_daily = ds_chl[chl_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
pom_daily = ds_pom[pom_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)
tpm_daily = ds_tpm[tpm_var_name].interp(time=time_horizon, latitude=ref_lats, longitude=ref_lons, kwargs=interp_kwargs)


# Merge into a single dataset
ds_daily = xr.Dataset({
    'salinity': sal_daily,
    'POC': poc_daily,
    'temperature': temp_daily,
    'Chl': chl_daily,
    'POM': pom_daily,
    'TPM': tpm_daily
})

print("Merged daily dataset")
ds_daily


Merged daily dataset


<xarray.Dataset> Size: 93MB
Dimensions:      (time: 60, latitude: 160, longitude: 220)
Coordinates:
    depth        float32 4B 0.0
  * time         (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
  * latitude     (latitude) float32 640B 30.12 30.38 30.62 ... 69.38 69.62 69.88
  * longitude    (longitude) float32 880B -12.88 -12.62 -12.38 ... 41.62 41.88
Data variables:
    salinity     (time, latitude, longitude) float64 17MB dask.array<chunksize=(10, 54, 220), meta=np.ndarray>
    POC          (time, latitude, longitude) float32 8MB dask.array<chunksize=(15, 80, 220), meta=np.ndarray>
    temperature  (time, latitude, longitude) float64 17MB dask.array<chunksize=(60, 160, 220), meta=np.ndarray>
    Chl          (time, latitude, longitude) float64 17MB dask.array<chunksize=(60, 160, 220), meta=np.ndarray>
    POM          (time, latitude, longitude) float64 17MB dask.array<chunksize=(60, 160, 220), meta=np.ndarray>
    TPM          (time, latitude, longitude) float64 17MB dask.array<chunksize=(60, 160, 220), meta=np.ndarray>

In [8]:
# del ds_poc  # Free memory immediately if needed 
# del ds_sal
# del ds_temp
# del ds_chl
# del ds_pom
# del ds_tpm
# gc.collect()

In [9]:
# Rechunk for optimal performance
# After all the merging and interpolating, Dask's chunks can get fragmented, rechunk so every Dask task receives 
# a data chunk of the exact size


print("Rechunking dataset for optimal performance")
ds_daily = ds_daily.chunk({
    'time': -1,       # Keep all time steps together
    'latitude': 80,   # Process 80 latitudes at once
    'longitude': 110  # Process 110 longitudes at once
})
print("Rechunked dataset")
print(ds_daily.chunks)

Rechunking dataset for optimal performance
Rechunked dataset
Frozen({'time': (60,), 'latitude': (80, 80), 'longitude': (110, 110)})


# ShellSIM model wraapper   
takes a 1D numpy array (time-series for one pixel) as input. run entire for loop (time-stepping) and return a 1D numpy array of the result ( eg soft tissue energy time-series). Then Apply in Parallel, using xr.apply_ufunc to apply wrapper function to the gridded data. tell apply_ufunc that the "core dimension" is time, which instructs it to parallelize over all other dimensions (lat, lon)

In [11]:

def run_fabm_at_point_full(T_ts, S_ts, Chl_ts, POC_ts, POM_ts, TPM_ts):
    """
    Runs FABM time-loop for a single spatial point.
    Returns a 1D array of output, e.g., Soft Tissue Energy.
    """
    # Check for NaN inputs
    if (np.any(np.isnan(T_ts)) or np.any(np.isnan(S_ts)) or 
        np.any(np.isnan(Chl_ts)) or np.any(np.isnan(POC_ts)) or 
        np.any(np.isnan(POM_ts)) or np.any(np.isnan(TPM_ts))):
        return np.full(time_horizon_len, np.nan)
    
    # Check for non-finite values
    if not (np.all(np.isfinite(T_ts)) and np.all(np.isfinite(S_ts)) and
            np.all(np.isfinite(Chl_ts)) and np.all(np.isfinite(POC_ts)) and
            np.all(np.isfinite(POM_ts)) and np.all(np.isfinite(TPM_ts))):
        return np.full(time_horizon_len, np.nan)
    
    try:
        # Initialize model
        model = pyfabm.Model("./fabm.yaml")
        
        # Set static dependencies
        model.cell_thickness = 1.0
        model.dependencies["seeding_rate"].value = 0.0
        model.dependencies["harvest_ratio"].value = 0.0
        model.dependencies["current_speed"].value = 1.0
        model.dependencies["air_exposure"].value = 0.0
        model.dependencies["number_of_days_since_start_of_the_year"].value = 0.0


        # Set INITIAL values for all dependencies
        # We use the first day's value (index 0) to initialize.
        model.dependencies["temperature"].value = float(T_ts[0])
        model.dependencies["practical_salinity"].value = float(S_ts[0])
        model.findStateVariable('Chl1/Chl').value = float(Chl_ts[0])
        model.findStateVariable('POC1/POC').value = float(POC_ts[0])
        model.findStateVariable('POM1/POM').value = float(POM_ts[0])
        model.findStateVariable('TPM1/TPM').value = float(TPM_ts[0])

        if not model.start():
            warnings.warn("!!! Model failed to start even after setting initial dependencies.")
            print(f"Model failed to start: {pyfabm.getError()}")
            return np.full(time_horizon_len, np.nan)
        
        # Initialize output
        outputs = np.zeros(time_horizon_len)
        
        # Daily integration
        for nd in range(time_horizon_len):
            # Set environmental variables
            model.dependencies["temperature"].value = float(T_ts[nd])
            model.dependencies["practical_salinity"].value = float(S_ts[nd])
            
            # Set food variables
            model.findStateVariable('Chl1/Chl').value = float(Chl_ts[nd])
            model.findStateVariable('POC1/POC').value = float(POC_ts[nd])
            model.findStateVariable('POM1/POM').value = float(POM_ts[nd])
            model.findStateVariable('TPM1/TPM').value = float(TPM_ts[nd])
            
            # Calculate and apply growth rates
            state_rates = model.getRates()
            model.state[:] += state_rates * 86400.
            
            # Save output (soft_tissue_output)
            outputs[nd] = model.state[0]

        return outputs
        
        #      # Save all 11 states
        #     outputs[:, nd] = model.state[:] 
        # # RETURN THE FULL 2D OUTPUT ARRAY
        # return outputs
        
        
    except Exception as e:
        warnings.warn(f"FABM error: {str(e)}")
        return np.full(time_horizon_len, np.nan)

## Most optimal processing method:  parallelize over the spatial dimensions (latitude, longitude) using xarray.apply_ufunc.  
chunking only the spatial dimensions (latitude and longitude) tells Dask to split the map into tiles, but keep the full time series for each pixel intact.

In [12]:
# apply_ufunc call. It accepts 6 distinct numpy arrays, one for each variable's time series at a single point.
# passes the 6 DataArrays from ds_daily to the 6 arguments of run_fabm_at_point_full
# Apply the model across the grid
# xarray.apply_ufunc: the clean way to run a function over each pixel (lat/lon) in parallel, using Dask underneath
# Dask will distribute this across threads or processes as needed.

print("Setting up parallel computation with apply_ufunc ...")

# result_sten = soft tissue energy 
result_sten = xr.apply_ufunc(
    run_fabm_at_point_full,
    
    # input data arrays
    ds_daily['temperature'],   
    ds_daily['salinity'],
    ds_daily['Chl'],
    ds_daily['POC'],
    ds_daily['POM'],
    ds_daily['TPM'],

    # function consumes 6 1D time array
    input_core_dims=[['time']] * 6,
    # outputs also with  a time dimension
    output_core_dims=[['time']],
    exclude_dims=set(('time',)),
    # use Dask parallel execution
    dask='parallelized',
    # run over all spatial dimensions
    vectorize=True,
    output_dtypes=[float],
    dask_gufunc_kwargs={
        'allow_rechunk': True,
        'output_sizes': {'time': time_horizon_len}
    }
)


# Add time coordinate back and set attributes
result_sten = result_sten.assign_coords(time=time_horizon)

# output rename for clarity 
result_sten = result_sten.rename('soft_tissue_energy')
result_sten.attrs['units'] = 'J'
result_sten.attrs['long_name'] = 'Oyster Soft Tissue Energy'

print("\nTask graph built. Ready to compute")
print(result_sten)

Setting up parallel computation with apply_ufunc ...

Task graph built. Ready to compute
<xarray.DataArray 'soft_tissue_energy' (latitude: 160, longitude: 220, time: 60)> Size: 17MB
dask.array<transpose, shape=(160, 220, 60), dtype=float64, chunksize=(80, 110, 60), chunktype=numpy.ndarray>
Coordinates:
    depth      float32 4B 0.0
  * latitude   (latitude) float32 640B 30.12 30.38 30.62 ... 69.38 69.62 69.88
  * longitude  (longitude) float32 880B -12.88 -12.62 -12.38 ... 41.62 41.88
  * time       (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
Attributes:
    units:      J
    long_name:  Oyster Soft Tissue Energy


# Run computation and Save  
Call .compute() or .to_netcdf() on the result. This triggers Dask to execute the parallel computation and write the final 3D output file.

In [ ]:
# # Compute with progress bar and save output when done
# output_file_name = "gridded_oyster_output.nc" 

# print("Now running dask computation ...")
# with ProgressBar():
#     result_sten.to_netcdf(output_file_name, compute=True)

# print(f" ****** SUCCESS: Results saved to {output_file_name} ******* ")

In [ ]:
# # Clean up
# del ds_poc  
# del ds_sal
# del ds_temp
# del ds_chl
# del ds_pom
# del ds_tpm
# del ds_daily, result_sten
# gc.collect()

# Method 2 on Memory optimization 

In [ ]:
time_horizon_len = ds_daily.time.size 
output_file_name = "gridded_oyster_output_soft_tissue_energy_batched.nc"


# 2. BATCHING LOOP
# Use  chunk sizes of  data
# Extract chunk definitions from the input data
# This assumes your ds_daily is already chunked (e.g., in Cell 3 of the original notebook)
try:
    lat_chunks = ds_daily.chunks['latitude']
    lon_chunks = ds_daily.chunks['longitude']
except KeyError:
    print("Error: ds_daily must be a Dask-backed xarray object with defined 'latitude' and 'longitude' chunks.")
    # Exit or provide placeholder logic if necessary
    raise

lat_indices = np.cumsum([0] + list(lat_chunks))
lon_indices = np.cumsum([0] + list(lon_chunks))

print(f"Starting batched computation and saving to {output_file_name}...")

# Loop over Latitude chunks
for i in range(len(lat_chunks)):
    lat_start = lat_indices[i]
    lat_end = lat_indices[i+1]
    
    # Loop over Longitude chunks
    for j in range(len(lon_chunks)):
        lon_start = lon_indices[j]
        lon_end = lon_indices[j+1]

        # Select the spatial subset (chunk) for computation
        ds_subset = ds_daily.isel(
            latitude=slice(lat_start, lat_end),
            longitude=slice(lon_start, lon_end)
        )
        
        print(f"Processing batch: Lat {i+1}/{len(lat_chunks)}, Lon {j+1}/{len(lon_chunks)}")
        
        # --- Run the user's original apply_ufunc logic on the subset ---
        result_sten_batch = xr.apply_ufunc(
            run_fabm_at_point_full,
            
            # input data arrays (subset)
            ds_subset['temperature'],   
            ds_subset['salinity'],
            ds_subset['Chl'],
            ds_subset['POC'],
            ds_subset['POM'],
            ds_subset['TPM'],

            input_core_dims=[['time']] * 6,
            output_core_dims=[['time']],
            exclude_dims=set(('time',)),
            dask='parallelized',
            vectorize=True, 
            output_dtypes=[float],
            dask_gufunc_kwargs={
                'allow_rechunk': True,
                'output_sizes': {'time': time_horizon_len}
            }
        )
        
        # Add time coordinate back and set attributes
        result_sten_batch = result_sten_batch.assign_coords(time=ds_subset.time)
        result_sten_batch = result_sten_batch.rename('soft_tissue_energy')
        result_sten_batch.attrs['units'] = 'J'
        result_sten_batch.attrs['long_name'] = 'Oyster Soft Tissue Energy'
        
        # --- Compute and Save (Force memory release for this batch) ---
        with ProgressBar():
            # Use 'w' (write) for the very first chunk (i=0, j=0) to create the file.
            # Use 'a' (append) for all subsequent chunks to add to the existing file.
            mode = 'w' if (i == 0 and j == 0) else 'a'
            
            # Convert to Dataset for cleaner NetCDF output/appending
            result_sten_batch.to_dataset().to_netcdf(
                output_file_name, 
                mode=mode, 
                format='NETCDF4', 
                compute=True
            )
            
print(f"✅ Success: All spatial batches processed and results saved incrementally to {output_file_name}")

In [ ]:
# # Getting all 11 states 
# N_STATES = 11
# STATE_NAMES = ['soft_tissue_energy', 'shell_energy', 'aging', 'C1', 'C2', 'C3', 'Chl_state', 'POC_state', 'POM_state', 'TPM_state', 'O2']

# # Apply the model across the grid
# result_full = xr.apply_ufunc(
#     run_pyfabm_model,
#     # --- INPUT data ---
#     ds_daily['temperature'], 
#     ds_daily['salinity'],
#     ds_daily['Chl'],
#     ds_daily['POC'],
#     ds_daily['POM'],
#     ds_daily['TPM'],

#     # Function now returns a 2D array (n_states, n_time_steps)
#     input_core_dims=[['time']] * 6,
#     output_core_dims=[['state'], ['time']], # New 'state' dimension
    
#     exclude_dims=set(('time',)),
#     dask='parallelized',
#     output_dtypes=[float],
#     dask_gufunc_kwargs={
#         'allow_rechunk': True,
#         # Define the size for the new 'state' dimension
#         'output_sizes': {'state': N_STATES, 'time': time_horizon_len}
#     }
# )

# # Rename the dimensions and coordinates
# result_full = result_full.rename({'state': 'state_variable', 'time': 'time'})
# result_full = result_full.assign_coords(state_variable=STATE_NAMES)
# result_full = result_full.assign_coords(time=time_horizon)
# result_full = result_full.to_dataset(dim='state_variable')

# print("Task graph built for all 11 states. Ready to compute")
# print(result_full)

# Validate on single pixel or tiny grid 

In [12]:

def check_subset_validity(ds_list, var_names, lat, lon, depth=0):
    """
    Checks if a single nearest grid point exists and contains valid data (not all NaNs) for the specified variables and depth.

    Args:
        ds_list (list): List of xarray Datasets (e.g., [ds_poc, ds_sal]).
        var_names (list): List of variable names corresponding to ds_list (e.g., ['poc', 'salinity']).
        lat (float): Latitude of the validation point.
        lon (float): Longitude of the validation point.
        depth (float): Depth level to check (e.g., 0.0 for surface).
    """
    print(f"Checking Validity for Lat: {lat}, Lon: {lon}")
    
    # Select the nearest point (and surface depth) from all datasets
    ds_point = ds_list[0].sel(
        latitude=lat, 
        longitude=lon, 
        depth=depth,
        method='nearest'
    )
    
    # Check if a point was even found (i.e., not an empty slice)
    if 'latitude' in ds_point.coords:
        found_lat = ds_point.latitude.item()
        found_lon = ds_point.longitude.item()
        print(f"Nearest grid point found at: Lat={found_lat:.2f}, Lon={found_lon:.2f}")
    else:
        print(f"⚠️WARNING: No grid point found near {lat}, {lon}. Check your grid bounds.")
        return

    all_valid = True
    
    for ds, var_name in zip(ds_list, var_names):
        # Select the specific variable and the single point
        # Need to re-select the variable from its own source file
        try:
            data_var = ds[var_name].sel(
                latitude=lat, 
                longitude=lon, 
                depth=depth,
                method='nearest'
            )
            # Load the 1D time series and check for all NaNs
            is_all_nan = np.isnan(data_var.compute()).all()
            
            if is_all_nan:
                print(f"⚠️WARNING: {var_name} is ALL NaN at this location. No model run possible.")
                all_valid = False
            else:
                print(f"✅ {var_name} contains valid data.")
        except Exception as e:
            print(f"❌ERROR: checking {var_name}: {e}")
            all_valid = False

    if all_valid:
        print("✅SUCCESS: All critical variables contain valid data at this location. Ready to run the model!")
    else:
        print("❌FAILED: One or more critical variables are missing data. Try a new location.")

# Validate if a given point exist in input variable data sources 

In [13]:


# # Coordinates for Gullmarsfjord, Sweden
# validation_lat = 58.2565
# validation_lon = 11.4444

# coordinates picked from the sample data source 
validation_lat = 45.125 
validation_lon = -7.875

# Run the check function
check_subset_validity(
    ds_list=[ds_poc, ds_sal], 
    var_names=[poc_var_name, salinity_var_name], 
    lat=validation_lat, 
    lon=validation_lon
)

Checking Validity for Lat: 45.125, Lon: -7.875
Nearest grid point found at: Lat=45.12, Lon=-7.88
✅ poc contains valid data.
✅ sos contains valid data.
✅SUCCESS: All critical variables contain valid data at this location. Ready to run the model!


In [14]:

# # Coordinates for Gullmarsfjord, Sweden
# validation_lat = 58.2565
# validation_lon = 11.4444

validation_lat = 45.125 
validation_lon = -7.875

print("Selecting single nearest grid point...")

# Select the single nearest grid point from the full dataset
ds_subset = ds_daily.sel(
    latitude=validation_lat, 
    longitude=validation_lon, 
    method='nearest'
)

# Load the single point data into memory
ds_subset.load()

print(" Validation point ready")
print(ds_subset)

Selecting single nearest grid point...
 Validation point ready
<xarray.Dataset> Size: 3kB
Dimensions:      (time: 60)
Coordinates:
    depth        float32 4B 0.0
  * time         (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
    latitude     float32 4B 45.12
    longitude    float32 4B -7.875
Data variables:
    salinity     (time) float64 480B 101.3 100.7 100.1 ... 68.89 68.33 67.76
    POC          (time) float64 480B 526.2 522.7 519.2 ... 324.5 321.0 317.4
    temperature  (time) float64 480B 0.3474 0.3158 0.3907 ... 0.6929 0.4366
    Chl          (time) float64 480B 0.565 0.5537 0.1136 ... 0.7708 0.6708 0.11
    POM          (time) float64 480B 0.7333 0.3554 0.3427 ... 0.46 0.5111 0.6623
    TPM          (time) float64 480B 0.5267 0.2862 0.5371 ... 0.4073 0.1479


In [15]:
# Run apply_ufunc on  Subset
result_sten_subset = xr.apply_ufunc(
    run_fabm_at_point_full,
    
    # ---  SUBSET data ---
    ds_subset['temperature'],   
    ds_subset['salinity'],
    ds_subset['Chl'],
    ds_subset['POC'],
    ds_subset['POM'],
    ds_subset['TPM'],

    input_core_dims=[['time']] * 6,
    output_core_dims=[['time']],
    exclude_dims=set(('time',)),
    dask='parallelized',
    output_dtypes=[float],
    dask_gufunc_kwargs={
        'allow_rechunk': True,
        'output_sizes': {'time': time_horizon_len}
    }
)


# Add time coordinate back and set attributes
result_sten_subset = result_sten_subset.assign_coords(time=time_horizon)
result_sten_subset = result_sten_subset.rename('soft_tissue_energy')
result_sten_subset.attrs['units'] = 'J'
result_sten_subset.attrs['long_name'] = 'Oyster Soft Tissue Energy'

print("Task graph built. Ready to compute subset.")
print(result_sten_subset)

Initializing Oyster...
   model type: shellsim_base
   3.7567048938207317        5.0000000000000000        3.1872102297907259     
   519.00531381912174        1599.3837221772937     
   initialization succeeded.
Initializing Chl1...
   model type: shellsim/prey
   initialization succeeded.
Initializing POC1...
   model type: shellsim/prey
   initialization succeeded.
Initializing POM1...
   model type: shellsim/prey
   initialization succeeded.
Initializing TPM1...
   model type: shellsim/prey
   initialization succeeded.
Initializing O2...
   model type: bb/passive
   initialization succeeded.
Task graph built. Ready to compute subset.
<xarray.DataArray 'soft_tissue_energy' (time: 60)> Size: 480B
array([1587.57848802, 1571.91663295, 1556.1691225 , 1540.47413326,
       1524.69229946, 1508.99922156, 1493.23609741, 1477.40240488,
       1461.63530552, 1445.87673426, 1430.0298525 , 1414.26520215,
       1398.33772504, 1382.44907492, 1366.41345716, 1350.55788768,
       1334.55655592, 13

In [16]:

# Compute with progress bar
print(" Running dask computation on subset...")

# output_file_name = "gridded_oyster_output_GULLMARSFJORD.nc"
output_file_name = "gridded_oyster_output_subsetpoint.nc"
with ProgressBar():
    # Save to a new file
    result_sten_subset.to_netcdf(
        output_file_name, 
        compute=True
    )

print(f"Success: Subset run done. Results saved to '{output_file_name}'")

 Running dask computation on subset...
Success: Subset run done. Results saved to 'gridded_oyster_output_subsetpoint.nc'


In [18]:
# # Validation on subset grid

# Coordinates for Gullmarsfjord, Sweden
validation_lat = 58.2565
validation_lon = 11.4444

# Define a wider 1.0 x 1.0 degree box.
lat_slice = slice(validation_lat - 0.5, validation_lat + 0.5)
lon_slice = slice(validation_lon - 0.5, validation_lon + 0.5)

# Run the check function
check_subset_validity(
    ds_list=[ds_poc, ds_sal], 
    var_names=[poc_var_name, salinity_var_name], 
    lat=validation_lat, 
    lon=validation_lon
)


Checking Validity for Lat: 58.2565, Lon: 11.4444
Nearest grid point found at: Lat=58.38, Lon=11.38
⚠️WARNING: poc is ALL NaN at this location. No model run possible.
✅ sos contains valid data.
❌FAILED: One or more critical variables are missing data. Try a new location.


In [19]:

print(f"Creating subset for lat: {lat_slice}, lon: {lon_slice}")
ds_subset = ds_daily.sel(
    latitude=lat_slice, 
    longitude=lon_slice
)

print("Loading small subset into memory...")
ds_subset.load()

print("Validation subset ready")
print(ds_subset)

Creating subset for lat: slice(57.7565, 58.7565, None), lon: slice(10.9444, 11.9444, None)
Loading small subset into memory...
Validation subset ready
<xarray.Dataset> Size: 47kB
Dimensions:      (time: 60, latitude: 4, longitude: 4)
Coordinates:
    depth        float32 4B 0.0
  * time         (time) datetime64[ns] 480B 2021-02-06 2021-02-07 ... 2021-04-06
  * latitude     (latitude) float32 16B 57.88 58.12 58.38 58.62
  * longitude    (longitude) float32 16B 11.12 11.38 11.62 11.88
Data variables:
    salinity     (time, latitude, longitude) float64 8kB 39.33 42.18 ... nan nan
    POC          (time, latitude, longitude) float64 8kB nan nan nan ... nan nan
    temperature  (time, latitude, longitude) float64 8kB -5.611e+03 ... 1.954...
    Chl          (time, latitude, longitude) float64 8kB -1.81e+03 ... 1.044e+04
    POM          (time, latitude, longitude) float64 8kB -1.433e+03 ... 1.013...
    TPM          (time, latitude, longitude) float64 8kB 430.5 ... -7.648e+03


In [20]:
# Run apply_ufunc on  Subset
result_sten_subset = xr.apply_ufunc(
    run_fabm_at_point_full,
    
    # ---  SUBSET data ---
    ds_subset['temperature'],   
    ds_subset['salinity'],
    ds_subset['Chl'],
    ds_subset['POC'],
    ds_subset['POM'],
    ds_subset['TPM'],

    input_core_dims=[['time']] * 6,
    output_core_dims=[['time']],
    exclude_dims=set(('time',)),
    dask='parallelized',
    output_dtypes=[float],
    dask_gufunc_kwargs={
        'allow_rechunk': True,
        'output_sizes': {'time': time_horizon_len}
    }
)


# Add time coordinate back and set attributes
result_sten_subset = result_sten_subset.assign_coords(time=time_horizon)
result_sten_subset = result_sten_subset.rename('soft_tissue_energy')
result_sten_subset.attrs['units'] = 'J'
result_sten_subset.attrs['long_name'] = 'Oyster Soft Tissue Energy'

print("Task graph built. Ready to compute subset.")
print(result_sten_subset)

ValueError: applied function returned data with an unexpected number of dimensions. Received 1 dimension(s) but expected 3 dimensions with names ('latitude', 'longitude', 'time'), from:

array([nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan, nan,
       nan, nan, nan, nan])

In [ ]:

# Compute with progress bar
print(" Running dask computation on subset...")

output_file_name = "gridded_oyster_output_GULLMARSFJORD.nc"

with ProgressBar():
    # Save to a new file
    result_sten_subset.to_netcdf(
        output_file_name, 
        compute=True
    )

print(f"Success: Subset run done. Results saved to '{output_file_name}'")